# 👕 IDM-VTON Virtual Try-On (ComfyUI Colab)
This notebook installs **IDM-VTON** which is excellent for high-fidelity garment details.

--- 
Developed by **nano** | Based on ComfyUI-On-Colab

In [ ]:
#@title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 2. Install ComfyUI & IDM-VTON Nodes
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install xformers -r requirements.txt

# Install Custom Nodes
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
!git clone https://github.com/shadowcz007/ComfyUI-IDM-VTON.git

# Install node requirements
%cd /content/ComfyUI/custom_nodes/ComfyUI-IDM-VTON
!pip install -r requirements.txt

In [ ]:
#@title 3. Download IDM-VTON Model Weights
%cd /content/ComfyUI/models/checkpoints
!apt-get -y install -qq aria2

# IDM-VTON Core Models
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/yisol/IDM-VTON/resolve/main/unet/diffusion_pytorch_model.bin -o idm_vton_unet.bin

# Base Stable Diffusion (if needed for workflow)
!wget -c https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt

In [ ]:
#@title 4. Launch ComfyUI
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0: break
      sock.close()
  print("\nComfyUI loaded, launching tunnel...\n")
  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("Your URL:", l[l.find("http"):])

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
%cd /content/ComfyUI
!python main.py --dont-print-server